# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

- **Record sets:** Tables or main data objects in the dataset.
- **Fields:** The columns for each record set, each with a unique `@id`.

Below we list the available record sets and their fields (referenced by `@id`).

In [ ]:
# List available record sets and their fields, referenced by @id
record_set_ids = []
print('Record sets and their fields:')
for record_set in metadata.record_sets:
    rec_id = record_set.id
    record_set_ids.append(rec_id)
    print(f"- RecordSet @id: {rec_id}  (name: {record_set.name})")
    print('  Fields:')
    for field in record_set.fields:
        print(f"    - Field @id: {field.id} (name: {field.name}, data type: {field.data_type})")
    print('')

## 3. Data Extraction
Load data from specific record sets into Pandas DataFrames for analysis. Refer to the `@id` of each record set.

In [ ]:
# Extract data from each record set into a dataframe
dataframes = {}
print('Available record sets:', record_set_ids)

# Choose the primary record set (use the first one for this notebook, change as needed)
if len(record_set_ids) == 0:
    raise Exception('No record sets found in the dataset schema.')
main_record_set_id = record_set_ids[0]

for rec_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rec_id))
    except Exception as e:
        print(f'Could not extract records for {rec_id}: {e}')
        continue
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f'RecordSet {rec_id} - Loaded {len(df)} rows')
    else:
        print(f'RecordSet {rec_id} has no rows or could not be loaded.')

# Show columns of the main record set
if main_record_set_id in dataframes:
    print('Fields (columns) in DataFrame:', dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print(f'No dataframe found for record set {main_record_set_id}.')

## 4. Exploratory Data Analysis (EDA)
You can now process, clean, and analyze the data. 
- Select a numeric field (`@id`) for analysis.
- Filter records, normalize data, and group by categorical fields. 
- Change `numeric_field_id` and `group_field_id` to actual IDs found in the DataFrame to suit your analysis.

In [ ]:
# Example: Choose numeric and group fields by their @id (update these with real values from overview)
# For demonstration, we probe for likely numeric columns
sample_df = dataframes[main_record_set_id]
numeric_candidates = sample_df.select_dtypes(include='number').columns.tolist()
print('Possible numeric columns:', numeric_candidates)

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # First numeric column
else:
    numeric_field_id = sample_df.columns[0]    # Just the first column (may need adjustment)
print('Chosen numeric field (by @id):', numeric_field_id)

# Threshold for filtering
threshold = sample_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(sample_df[numeric_field_id]) else None

# Filtering records
if threshold is not None:
    filtered_df = sample_df[sample_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)}")
else:
    print('Chosen field is not numeric, cannot filter.')

# Normalization
if threshold is not None:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized field {numeric_field_id} (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by most frequent non-numeric field
non_numeric = [c for c in sample_df.columns if sample_df[c].dtype == 'object']
group_field_id = non_numeric[0] if non_numeric else None
print('Chosen group field (by @id):', group_field_id)

if group_field_id and pd.api.types.is_numeric_dtype(sample_df[numeric_field_id]):
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())
else:
    print('No suitable group field found or numeric field is not numeric.')

## 5. Visualization
Visualize data distributions or relationships between fields. You can further customize this based on the field `@id`s available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if threshold is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(sample_df[numeric_field_id], kde=True, bins=30)
    plt.title(f'Distribution of field: {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(12,5))
        order = sample_df[group_field_id].value_counts().index[:10]
        sns.boxplot(data=sample_df, x=group_field_id, y=numeric_field_id, order=order)
        plt.xticks(rotation=45, ha='right')
        plt.title(f'{numeric_field_id} by {group_field_id} (top 10)')
        plt.show()
else:
    print('No numeric field found for plotting.')

## 6. Conclusion
This notebook provided an overview of the FAIR² Mass Spectrometry dataset using the `mlcroissant` library. We demonstrated how to list record sets and fields by their `@id`, extract tabular data into Pandas DataFrames, perform basic cleaning, normalization, and grouping operations, and visualize field distributions. For deeper biological or mass spec investigation, update the chosen fields (by `@id`) as appropriate for your research.

For more information and further analysis, refer to the original dataset publication and documentation.